# ViInfographicVQA — zero-shot baseline (single-image task)

Runs a Qwen2.5-VL model on the single-image test split (1,374 questions, 201 infographics)
with the same prompt, image resolution and scorer as the paper
([arXiv 2512.12424](https://arxiv.org/abs/2512.12424)), so the numbers are comparable.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Progress is saved every 25 questions; re-running the notebook resumes where it stopped.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"   # paper baseline: "Qwen/Qwen2.5-VL-7B-Instruct"
LIMIT = None            # e.g. 50 for a quick trial run; None = the whole test split
SAVE_TO_DRIVE = True    # keep results on Google Drive so a disconnect loses nothing
LOAD_IN_4BIT = False    # set True for the 7B model on a 16 GB GPU

# Same image budget as the paper's Qwen wrapper.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 100

DATASET = "duytranus/ViInfographicVQA"
SPLIT_FILE = "data/single_test.json"

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils Levenshtein bitsandbytes

In [ ]:
import json, os
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/viinfographicvqa/results")
else:
    OUT_DIR = Path("/content/results")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / f"single_test__{MODEL_ID.split('/')[-1]}.json"
print("results ->", OUT_FILE)

## Data: annotations plus only the images the test split needs (~250 MB)

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

DATA_DIR = Path("/content/ViInfographicVQA")
hf_hub_download(DATASET, SPLIT_FILE, repo_type="dataset", local_dir=DATA_DIR)
rows = json.loads((DATA_DIR / SPLIT_FILE).read_text(encoding="utf-8"))
if LIMIT:
    rows = rows[:LIMIT]
images = sorted({r["image_path"] for r in rows})
snapshot_download(DATASET, repo_type="dataset", local_dir=DATA_DIR, allow_patterns=images)
missing = [p for p in images if not (DATA_DIR / p).exists()]
assert not missing, f"{len(missing)} images failed to download, e.g. {missing[:3]}"
print(len(rows), "questions on", len(images), "images")

## Model

In [ ]:
import torch
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
kwargs = dict(torch_dtype=torch.bfloat16, device_map="cuda")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, use_fast=True)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

## Inference

Prompt and answer parsing are copied from the paper's code. One deliberate difference:
decoding is greedy (`do_sample=False`) so results are reproducible.

In [ ]:
from tqdm.auto import tqdm

SYSTEM_PROMPT = ("Answer the following question based solely on the image content "
                 "concisely with a single term.")


def parse_answer(response: str) -> str:
    return response.split("Answer:")[-1].strip() if "Answer:" in response else response.strip()


@torch.no_grad()
def infer(question: str, image_path: str) -> str:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text", "text": f"Question: {question}\nAnswer:"},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return parse_answer(processor.batch_decode(new_tokens, skip_special_tokens=True)[0])


done = {}
if OUT_FILE.exists():
    done = {r["question_id"]: r["predict"] for r in json.loads(OUT_FILE.read_text(encoding="utf-8"))
            if "predict" in r}
    print("resuming:", len(done), "answers already saved")


def save():
    OUT_FILE.write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")


for i, row in enumerate(tqdm(rows)):
    if row["question_id"] in done:
        row["predict"] = done[row["question_id"]]
        continue
    row["predict"] = infer(row["question"], str(DATA_DIR / row["image_path"]))
    if i % 25 == 0:
        save()
save()
print("saved", OUT_FILE)

## Score

`ANLS` reproduces the paper's scorer exactly; `ANLS-std` is the textbook definition; `Acc` is exact match.
Paper, single-image overall ANLS: Qwen2.5-VL-7B 67.42 · fine-tuned 67.80 · InternVL3.5-8B 67.02 · Ovis2.5-9B 71.02.

In [ ]:
from collections import defaultdict

import Levenshtein

THRESHOLD = 0.5


def clean_prediction(text: str) -> str:
    return text.lower().strip().rstrip(".").replace('"', "").rstrip(">").lstrip("<")


def clean_answer(text: str) -> str:
    return text.lower().strip()


def paper_anls(answer: str, prediction: str) -> float:
    p = clean_prediction(prediction).replace('"', "").rstrip(".").lower()
    score = Levenshtein.ratio(p, clean_answer(answer))
    return score if score >= THRESHOLD else 0.0


def standard_anls(answer: str, prediction: str) -> float:
    p, g = clean_prediction(prediction), clean_answer(answer)
    if not p and not g:
        return 1.0
    nl = Levenshtein.distance(p, g) / max(len(p), len(g))
    return 1.0 - nl if nl < THRESHOLD else 0.0


def exact_match(answer: str, prediction: str) -> float:
    return float(clean_prediction(prediction) == clean_answer(answer))


METRICS = {"anls": paper_anls, "anls_std": standard_anls, "acc": exact_match}

# Fields to break results down by; whichever exist in the split are used.
GROUP_FIELDS = ("answer_source", "element", "image_type")


def _as_labels(value) -> list[str]:
    """`element` is a string in most rows but a list in a few."""
    if value is None:
        return []
    return [str(v) for v in value] if isinstance(value, list) else [str(value)]


def score(rows: list[dict], pred_key: str = "predict") -> dict:
    """Return {"overall": {...}, "<field>": {"<label>": {...}}} with scores in percent."""
    sums: dict = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    for row in rows:
        vals = {name: fn(row["answer"], str(row.get(pred_key, ""))) for name, fn in METRICS.items()}
        groups = [("overall", "overall")]
        groups += [(f, label) for f in GROUP_FIELDS for label in _as_labels(row.get(f))]
        for field, label in groups:
            bucket = sums[field][label]
            bucket["count"] += 1
            for name, v in vals.items():
                bucket[name] += v
    out: dict = {}
    for field, labels in sums.items():
        table = {label: {"count": int(b["count"]),
                         **{m: round(100 * b[m] / b["count"], 2) for m in METRICS}}
                 for label, b in labels.items()}
        out[field] = table["overall"] if field == "overall" else table
    return out


def format_report(result: dict) -> str:
    lines = [f"{'':32s}{'n':>6s}{'ANLS':>8s}{'ANLS-std':>10s}{'Acc':>8s}"]

    def row(name: str, s: dict) -> str:
        return f"{name[:31]:32s}{s['count']:6d}{s['anls']:8.2f}{s['anls_std']:10.2f}{s['acc']:8.2f}"

    lines.append(row("Overall", result["overall"]))
    for field in GROUP_FIELDS:
        if field in result:
            lines.append(f"-- {field}")
            for label, s in sorted(result[field].items(), key=lambda kv: -kv[1]["count"]):
                lines.append(row("  " + label, s))
    return "\n".join(lines)


print(MODEL_ID)
print(format_report(score(rows)))

In [ ]:
# A few wrong answers, to see what kind of mistakes the model makes.
wrong = [r for r in rows if paper_anls(r["answer"], r["predict"]) == 0][:15]
for r in wrong:
    print(f"[{r['answer_source']} | {r['element']}] {r['question']}\n   gold: {r['answer']}\n   pred: {r['predict']}\n")